In [2]:
#export 

import logging 
import math 
from openai import OpenAI 

import pandas as pd
import os 
import json 

import ollama_manager


/Users/zzddfge/.pyenv/versions/mlx-31211/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
This is a beta version of the video understanding. It may not work as expected.


In [3]:
#export 

########################################
#### Asignación de prompts
########################################

prompt1_todas = None
prompt2_incidencia = None
prompt3_propuesta = None

PROMPT_TODOS = "Todos"
PROMPT_INCIDENCIAS = "Incidencia"
PROMPT_PROPUESTA = "Propuesta"

def __set_prompt(tipo_prompt, prompt):
    global prompt2_incidencia
    global prompt1_todas

    if tipo_prompt == PROMPT_TODOS:
        prompt1_todas = prompt 
    elif tipo_prompt == PROMPT_INCIDENCIAS:
        prompt2_incidencia =  prompt
        
def __get_prompt(tipo_prompt):
    global prompt2_incidencia
    global prompt1_todas
    prompt = None 

    if tipo_prompt == PROMPT_TODOS:
        prompt = prompt1_todas
    elif tipo_prompt == PROMPT_INCIDENCIAS:
        prompt = prompt2_incidencia 

    return prompt

def __get_tokens(prompt:str):
    palabras = prompt.split()
    tokens=math.ceil(len(palabras)*1.35)

    return palabras, tokens

In [4]:
#export 

########################################
#### FUNCIONES PARA CARGA DE INCIDENCIAS (EXCEL DE ICT)
########################################

#######
## PARTE 1 - Lectura del Excel 
def getIncidencias(url, fichero, lista_campos = ["Número", "Descripción corta", "Notas de resolución"]):
    """
    Lee un archivo de incidencias Excel y se queda solo con el listado de campos que se especifican. 
    La función sólo está preparada para procesar 3 campos:
    - Número
    - Descripción
    - Notas de resolución

    Este debe ser el orden de los campos. 

    Sólo da libertad para poder cargar diferentes excels que tengan nombre de campos distintos de forma paramétrica.

    Parametros:
        url: URL del sitio web donde se encuentra el archivo Excel.
        fichero: Nombre del archivo Excel.  
        lista_campos: Lista de campos que se quieren extraer del archivo Excel. Por defecto, se extraen los campos "Número", "Descripción corta" y "Notas de resolución".
    
    Return:
        Un DataFrame con los campos especificados extraídos del archivo Excel.
    """
    method ="getIncidencias"

    file_name_path = f"{url}/{fichero}"
    logging.info(f"{method}-START - Leyendo el archivo Excel: {file_name_path}")

    df = pd.read_excel(file_name_path)
    logging.info(f"{method}- {df.shape[0]} registros leídos.")

    #Copia y selección de los tres campos que necesitamos.
    df_filtrado = df.copy()
    df_filtrado = df_filtrado[lista_campos]
    df_filtrado.columns = df_filtrado.columns.str.lower()
    df_filtrado.rename(columns={lista_campos[0].lower(): "id"
                                , lista_campos[1].lower(): "desc"
                                , lista_campos[2].lower(): "res"
                                }, inplace=True)
    
    #Creación del listado json antes de añadirle el campo status.
    listado_json = df_filtrado.to_json(orient='records')

    #Creación de una nueva columna de estado a PENDIENTE.
    df_filtrado["status"] = "PEND"
    df_filtrado["cot"] = ""

    logging.debug(f"{method}-END - Se ha filtrado el DataFrame para obtener los campos especificados: {df_filtrado.columns}")

    return listado_json, df_filtrado


#######
## PARTE 2 - Aplicaciones disponibles
def get_lista_aplicaciones():
    lista_app=["OnBoarding", "Analytics"]

    return lista_app





In [5]:
#export 

#######
## CONSOLIDANDO LA INFORMACION DE LOS 
## PARTE 2 - 1. preparación de PROMPTS PARA CADA TICKETS

In [6]:
#export 

#######
## CONSIDERANDO SOLO 1 LLAMADA por ticket
## PARTE 2 - 1. preparación de PROMPTS PARA CADA TICKETS


def get_causa_raiz(desc, res, model="gemma3:27b", thinking=False, temperature=0.7):
    """ 
    La función get_causa_raiz recibe una descripción de un incidente (desc) y una nota de resolución y a través
    de la invocación de un modelo LLM que se indica, ejecuta una clasificación del tipo de incidencia, 
    investiga la causa raíz de la misma y propone una solución viable para el mismo. También genera una 
    parte más detallada del tipo de información que falta en el caso que falte información. 

    La propuesta de solución es un poco genérica, pero se puede empezar con esta información. Los prompts son en 
    inglés porque se tiene en general un poco mejor de precisión. 

    Parametros:
    - desc: str Descripción de la incidencia dada de alta por el usuario.
    - res: str Notas de resolución dada por el proveedor. 
    - model: str (default="gemma3:27b") Modelo LLM a usar para generar la información. 
    - thinking: bool (default=False) Indicativo si se requiere activar o no el modelo thinking en caso que el modelo permita no activarlo.
    - temperature: float (default=0.7) Temperatura para la generación de texto por el modelo LLM.

    Returns:
    - El modelo se ha configurado para poder devolver la salida en un json con tres campos: rootcause, solution, data_missing.
      Devuelve la cadena de razonamiento por un lado, si la hay, y el contenido sin la marca que sea un json por otro.
    """

    method = "get_causa_raiz"
    system = ''' 
    You are an expert analist in detecting the root cause for incidents in the department of micro computer. 
    You works for a very big company first of its class. 
    You are the best detecting root cause and proposing viable solutions. 
    You have a reward of 2.000.000$ for reducing number of incidents with your solutions. 
    If cant make proposals, you are fired. 
    The only lenguage you can answered is spanish whatever language you can find in the incidents. 
    '''
    user = f"""
    #CONTEXT.
    You are a top-tier incident analyst focused on reducing the number of tickets. 
    You are contracted to analyze the main root cause of each incident and propose a concrete fix.  
    Each incident provides three fields:
    • id: incident code.
	• desc: problem description.
	• res: resolution notes for the ticket.

    #TASK.
     1. From one incident, identify the main problem(s) and propose a viable, actionable solution.
     2. Detect these groups/conditions ONLY when explicitly supported by the text:
     2.1. Group 1 (personal data missing): DNI, birthdate, email, telephone number.
     2.2. Group 2 (request to change process data): e.g., start/end dates, email, etc.
     2.3. Group 3 (onboarding workflow not progressing) due to any of:
     - User not logged ina) Tasks not finished (e.g., Draft, Ready, etc.)
     - Missing/Inactive license
     - User not active in the Enelint domain
     - User cannot upload information to the portal
     2.4. Group 4 (no access) due lacks access to the portal or to another application.
     3. Classify the root cause with ONE label from “CLASSIFICATION OPTIONS”.
     4. Output ONLY a JSON object with EXACTLY these 12 fields, in this exact order: classification, rootcause, solution, data_missing, personal_data, request_modification, workflow_stopped, previous_task_active, licence_inactive, upload_information, access_app_error, task_stopped

     #FIELD RULES (STRICT)
     - Language: All JSON string values must be in Spanish.
     - rootcause: max 10 words; concise summary of the cause.
     - solution: specific, practical action the analyst would take or request.
     - data_missing: array of attributes missing or incorrect (e.g., "DNI", "email", "fecha de nacimiento", "teléfono", or other attributes explicitly mentioned). Use [] if none.
     - personal_data, request_modification, workflow_stopped, previous_task_active, licence_inactive, upload_information, access_app_error: each must be "S" or "N" only. Use "S" ONLY if there is explicit evidence in desc or res; otherwise "N".
     - task_stopped: array with names of unfinished previous tasks EXACTLY as found in the text when the workflow is stopped. Extract names that appear after the literal prefix "Task " (example: "Task Verificación de Identidad"). If multiple, include all. Use [] if none.

     #DECISION RULES
     - Evidence over inference: mark "S" only with clear evidence; otherwise "N".
     - If multiple classifications could apply, choose the single most specific using this priority:
     1) Grupo 1
     2) Grupo 2
     3) Grupo 3
     4) Grupo 4
     5) Faltan datos
     6) Datos incorrectos
     7) Problemas VPN
     8) Problemas Teams
     9) Problemas de conexión
     10) Problemas de permisos
     11) Problemas de software12) Problemas de hardware
     13) Problemas de configuración
     14) Otros
     - Unknown/insufficient info: if details are insufficient, set flags to "N", arrays to [], classification to "Otros", rootcause to "Información insuficiente", and propose a solution that requests the exact missing data.

    #CLASSIFICATION DETECTION RULES (for 5–14)
    5) Faltan datos — Problemas por datos faltantes (no personales)
       Match when: required non-personal info is missing (documents, case IDs, attachments, forms).
       Keywords: "falta", "no adjunta", "no aporta", "incompleto", "sin documento", "missing attachment", "not provided", "no indica", "campo vacío".
       Exclude: personal data → that is Grupo 1.
    
    6) Datos incorrectos — Problemas por datos erróneos
       Match when: data present but wrong/invalid/inconsistent.
       Keywords: "erróneo", "incorrecto", "equivocado", "inconsistente", "formato inválido", "mal cargado", "mismatch", "typo".
       Examples: email mal escrito, fecha inválida, número fuera de rango.

    7) Problemas VPN — Incidencias por la VPN
       Match when: failure tied to VPN usage.
       Keywords/brands: "VPN", "GlobalProtect", "AnyConnect", "FortiClient", "túnel", "split tunnel", "cuando conecto la VPN", "se corta la VPN", "no levanta túnel".
       Prefer over generic connection issues (priority 7 > 9).
    
    8) Problemas Teams — Microsoft Teams
       Match when: issue mentions Microsoft Teams (app, calls, chat, meetings).
       Keywords: "Teams", "Microsoft Teams", "no inicia Teams", "no puedo unirme", "no sincroniza", "no recibe llamadas", "grabación Teams".
    
    9) Problemas de conexión — Red (no VPN)
       Match when: connectivity/latency/DNS/proxy/timeouts not tied to VPN.
       Keywords: "sin internet", "caída de red", "latencia", "no resuelve DNS", "timeout", "proxy", "no carga página", "packet loss", "ping falla".
       Exclude if VPN is explicitly involved → use Problemas VPN.
    
    10) Problemas de permisos — Roles/visibilidad/autorización dentro de apps
        Match when: user enters the app but lacks rights/visibility or sees permission errors.
        Keywords: "sin permisos", "permiso denegado", "no le aparece", "no ve datos", "rol", "perfil", "RBAC", "autorización", "alta de permisos", "asignar rol".
        Heuristic: Whole-app/portal access blocked → Grupo 4. Access ok but missing rights → Problemas de permisos.
    
    11) Problemas de software — Fallos genéricos de software
        Match when: application bug/crash/installation/compatibility not specific to Teams or configuration.
        Keywords: "bug", "error de aplicación", "se cierra", "crashea", "exception", "stacktrace", "no instala", "incompatible", "actualización fallida".
        Exclude: if clearly config file/parameter issue → Problemas de configuración (13).
    
    12) Problemas de hardware — Dispositivo físico
        Match when: physical device failure.
        Keywords: "teclado", "ratón", "monitor", "impresora", "portátil", "batería", "no enciende", "dañado", "golpe", "sobrecalentamiento", "LED parpadea", "ruido disco".
    
    13) Problemas de configuración — Parámetros/archivos/policies
        Match when: wrong config, file, path, environment variable, registry, policy, DNS/proxy settings explicitly misconfigured.
        Keywords: "configuración", "parámetro", "ruta", "variable de entorno", "registro", "policy", "GPO", ".ini", ".conf", "yml/json", "proxy mal configurado", "DNS mal configurado".
        Prefer this over generic software if config is the root cause.
    
    14) Otros — Cualquier caso no cubierto
        Use only when none of the above (or Grupo 1–4) match with evidence.
    
    #BOOLEAN FLAGS CONSISTENCY (mapping hints)
    - access_app_error = "S" when there is inability to access a system/app (Grupo 4) OR explicit permission/auth errors (Problemas de permisos). Otherwise "N".
    - workflow_stopped / previous_task_active / licence_inactive / upload_information: set according to onboarding evidence only (Grupo 3 subcauses).
    - personal_data = "S" only for missing/invalid personal data (Grupo 1).
    - request_modification = "S" when the user requests changing process data (Grupo 2).
    - data_missing: list only attributes explicitly missing/incorrect (non-personal go to this list; personal data also list them here when Grupo 1 applies).
    
    #CLASSIFICATION OPTIONS (exact strings; do not translate or alter)
    1. Grupo 1 — Faltan datos personales (DNI, fecha de nacimiento, email, teléfono).
    2. Grupo 2 — Solicitud de cambio de datos del proceso (fechas de inicio/fin, email, etc.).
    3. Grupo 3 — El workflow de onboarding no progresa.
    4. Grupo 4 — Sin acceso al portal u otra aplicación.
    5. Faltan datos — Problemas por datos faltantes (en general).
    6. Datos incorrectos — Problemas por datos erróneos.
    7. Problemas VPN — Incidencias de conexión debidas a la VPN.
    8. Problemas Teams — Incidencias con la aplicación Microsoft Teams.
    9. Problemas de conexión — Incidencias de red no relacionadas con la VPN.
    10. Problemas de permisos — Falta de permisos, acceso o visibilidad en aplicaciones/sistemas.
    11. Problemas de software — Fallos genéricos de software.
    12. Problemas de hardware — Fallos de dispositivos físicos.
    13. Problemas de configuración — Errores en archivos o parámetros de configuración.
    14. Otros — Cualquier otro caso no cubierto.

    #QUALITY CHECK (before finalizing)
    - Ensure all strings are in Spanish.
    - Ensure the 12 fields are present and in the exact order.
    - Ensure only "S" or "N" are used for the specified boolean fields.
    - Ensure rootcause has 10 words or fewer.
    - Ensure arrays use [] when empty and contain exact task names after "Task ".

    #INPUT
    The incident description is: {desc}
    The resolution notes are: {res}
    
    #FINAL OUTPUT FORMAT (return only this JSON; example structure shown, replace values accordingly)
    {{
        "classification": "...","rootcause": "...","solution": "...","data_missing": [],"personal_data": "N",
        "request_modification": "N","workflow_stopped": "N","previous_task_active": "N","licence_inactive": "N",
        "upload_information": "N","access_app_error": "N","task_stopped": []
    }}
    """

    prompt_user = __get_prompt(PROMPT_INCIDENCIAS)
    if prompt_user is not None:
        logging.debug(f"{method}- prompt= {prompt_user}")
        user=prompt_user.format(desc=desc, res=res)

    palabras, num_tokens = __get_tokens(user)
    logging.debug(f"{method}-START ejecutando modelo {model} para la descripción {desc} y nota de resolución {res}")
    
    resp = ollama_manager.call_ollama_api(model=model, system_message=system, user_message=user, temperature=temperature)
    resp = resp.lower().replace("json", '').replace("```", '').replace("\n", '')
    content, cot = ollama_manager.separa_thinking_contenido(resp)
    
    return content, cot, num_tokens

def add_columns(df, fichero_output_path, model="gemma3:latest", thinking=False, temperature=0.7):
    """ 
    Agrega columnas a un DataFrame llamado df con las causas raíz y clasificaciones obtenidas por el modelo Ollama.
    
    Parametros:
           df (pandas.DataFrame): El DataFrame que contiene las descripciones y notas de resolución.
           model (str): El nombre del modelo Ollama a usar para obtener las causas raíz y clasificaciones. Por defecto, es "gemma3:latest".
           thinking (bool): Indica si se debe considerar el pensamiento en la clasificación. Por defecto, es False.

    Return:
        Una copia del data frame original con las nuevas columnas para todas las incidencias.
    """
    method="add_columns"

    logging.info(f"{method}-START iniciando generación información para {df.shape[0]} registos con el modelo {model} y cot {thinking}")

    df_nuevo = df.copy()
    generado=0
    for index, row in df_nuevo.iterrows():
        desc = row['desc']
        res = row['res']
        status = row['status']

        #Sólo procesamos los datos con la IA si no están ya generados correctamente en la fuente.
        if status != 'OK':
            content, cot, num_tokens = get_causa_raiz(desc, res, model, thinking, temperature)
        
            logging.info(f"{method}- Index={index}, ID = {row['id']} con {num_tokens} tokens --> rootcause={content}")

            try:
                contenido = json.loads(content)
                l = list(contenido.items())
                k0, clas = l[0]
                k1, root = l[1]
                k2, solution = l[2]
                """ 
                k3, data_missing = l[3]

                k4, personal_data = l[4]
                k5, request_modification = l[5]
                k6, workflow_stopped = l[6]
                k7, previous_task_active = l[7]
                k8, licence_inactive = l[8]
                k9, upload_information = l[9]
                k10, access_app_error = l[10]
                k11, task_stopped = l[11]
                """ 
                df_nuevo.at[index, 'clasification'] = clas
                df_nuevo.at[index, 'root_cause'] = root
                df_nuevo.at[index, 'solution'] = solution
                df_nuevo.at[index, 'cot'] = cot

                """ 
                df_nuevo.at[index, 'data_missing'] = f"{data_missing}"
                df_nuevo.at[index, 'personal_data'] = f"{personal_data}"
                df_nuevo.at[index, 'request_modification'] = f"{request_modification}"
                df_nuevo.at[index, 'workflow_stopped'] = f"{workflow_stopped}"
                df_nuevo.at[index, 'previous_task_active'] = f"{previous_task_active}"
                df_nuevo.at[index, 'licence_inactive'] = f"{licence_inactive}"
                df_nuevo.at[index, 'upload_information'] = f"{upload_information}"
                df_nuevo.at[index, 'access_app_error'] = f"{access_app_error}"
                df_nuevo.at[index, 'task_stopped'] = f"{task_stopped}"
                """

                df_nuevo.at[index, 'status'] = "OK"
            except Exception as e:
                logging.error(f"{method}-ERROR al parsear JSON con error {e}")
                df_nuevo.at[index, 'status'] = "ERROR"
            
            #Cada 50 interacciones guardamos el fichero para ir manteniendo el progreso.
            generado+=1
            if generado % 20 == 0:
                df_nuevo.to_excel(fichero_output_path)
                logging.info(f"{method}- Guardando archivo Excel en {fichero_output_path}")

        else:
            logging.info(f"{method}- Index={index}, Status = OK para el id {row['id']}")
            

    return df_nuevo


In [7]:
#export

def procesa_incidencias(df, url, fichero, model, thinking, temperature):
    """ 
    Procesa las incidencias para obtener la clasificación, raíz de causa y solución.

    """
    method="procesa_incidencias"

    #Generamos el nombre del fichero de salida.
    fichero_ouput=f"{fichero}_{model.replace(':', '_')}.xlsx"
    fichero_ouput_path=os.path.join(url, fichero_ouput)

    #Si el fichero existe, lo cargamos en vez de procesar los datos que nos llegan. 
    #Si no, procesamos los datos que nos llegan. 
    if os.path.exists(fichero_ouput_path):
        df_nuevo = pd.read_excel(fichero_ouput_path)
        logging.info(f"{method}-START Fichero {fichero_ouput_path} ya existente. Se ha leído.")
    else:
        df_nuevo = df.copy()
        logging.info(f"{method}-START Procesando por primera vez al no existir {fichero_ouput_path}.")
    
    #Procesa cada incidencia.
    df_nuevo = add_columns(df_nuevo, fichero_ouput_path, model=model, thinking=thinking, temperature=temperature)

    #Genera un archivo de salida Excel con el nombre del modelo, reemplazando el : por _.
    df_nuevo.to_excel(fichero_ouput_path)
    
    logging.info(f"{method}- Archivo de salida generado: {fichero_ouput}")

    
    

In [8]:
#export 

#######
## CONSIDERANDO SOLO 1 LLAMADA PARA TODAS LAS INCIDENCIAS
## PARTE 1 - 1. Preparación de PROMPTS

def analiza_todas_incidencias(listado_json, url, fichero, app="", model="gemma3:latest", thinking=False, temperature=0.7):
    system = ''' 
        You are an expert analist in detecting the root cause for incidents in the department of micro computer. 
        You works for a very big company first of its class. 
        You are the best detecting root cause and proposing viable solutions. 
        You have a reward of 2.000.000$ for reducing number of incidents with your solutions. 
        If cant make proposals, you are fired. 
        The only lenguage you can answered is spanish whatever language you can find in the incidents. 
        '''
    
    user=f""" 
    #ROLE
    You are a top-class incident analyst, focused on reducing the number of tickets. 
    You have been hired to identify the main root cause of incidents and propose viable solutions that address these causes. 
    The user will provide you with a list of incidents in JSON format.
    The name of the application is {app}

    #INPUT FORMAT (JSON)
    The list is an array of objects with the following fields:
	• id: incident code.
	• desc: problem description.
	• res: resolution notes for the ticket.

    Example input (referred later as incident_list_json:
    [{{"id":"INC-001","desc":"Description...","res":"Notes..."}},{{"id":"INC-002","desc":"Description...","res":"Notes..."}}]
    
    #TASK
	1. Identify the main root cause for each incident.
	2. From these root causes, group common problems and propose viable solutions for each common problem.
	3. Prepare a list of common problems that includes the list of incident IDs for each problem.
	4. Respond only in the exact structure described below (A–D), using a professional tone, and in Spanish.

    #ANALYSIS RULES
	• If an incident seems to have multiple causes, choose only one main root cause (the most decisive).
	• If information is missing, infer the best possible root cause and mark it as “low confidence” when applicable.
	• Normalize wording to group similar causes (e.g., “VPN down”, “VPN error”, “VPN connectivity failure” ⇒ one single problem).
	• Include all incidents provided in the analysis and in the counts.
	• Calculate statistics for root causes (percentage of total incidents). Round to one decimal place.
	• Do not invent data. If some fields (desc or res) are missing, state it and proceed with what is available.

    #REQUIRED OUTPUT FORMAT
    Respond only with the following sections, in order, without any extra text:
    
    A. NUMBER OF INCIDENTS.
	• State the total number of incidents received.
    
    B. INCIDENT ANALYSIS.
	• B.1 Detailed analysis for each root cause: main root cause, and short justification (mention signals from desc/res).
	• B.2 Root cause statistics: list/table showing each cause and its percentage of total incidents.
	• B.3 Proposed solutions for each common problem: for each problem/root cause, provide specific and viable actions (e.g., fixes, prevention, automation, monitoring, documentation, training, process changes).

    C. ROOT CAUSE LIST.
	• Return only a JSON array with this schema for each item:
	• "problema": normalized name of the common problem/root cause.
	• "incidencias": array with the id values of the incidents belonging to that problem.
    
    Example (for reference):
    [ {{"problema":"Error de conectividad con VPN","incidencias":["cod1","cod2","cod3"]}}, 
      {{"problema":"Credenciales inválidas SSO","incidencias":["cod4"]}}
    ]

    D. CONCLUSIONS.
	• Clear, actionable conclusions (e.g., quick wins, risks, priorities, follow-up metrics).
    
    E. VERIFICATION BEFORE ANSWERING
	• Make sure all incidents from incident_list_json are present in sections A–C.
	• Recount the total number and confirm it matches the value in section A.
	• Ensure the final answer is entirely in Spanish; if not, translate it into Spanish before returning it.

    #INPUT / inciden_list_json
    The incident list is: {listado_json}
    """
    method="analiza_todas_incidencias"

    #Verificamos si llega informada el nombre de la aplicación.
    cadena_app=""
    cadena_app2=""
    if app is not None and app != "":
        cadena_app = f"The incidents are from the Application {app}"
        cadena_app = f"B.3 Add the total number of incidents and the percentage of them that the incident is classified in a wrong way for the application {app}"
        
    user.replace("*APP*", cadena_app)
    user.replace("*APP2*", cadena_app2)

    #Generamos el nombre del fichero de salida.
    fichero_ouput=f"{fichero}_{model.replace(':', '_')}.txt"
    fichero_ouput_path=os.path.join(url, fichero_ouput)

    #Si ya existe no hacemos nada.
    if os.path.exists(fichero_ouput_path):
        logging.info(f"{method}-START fichero {fichero_ouput} ya existe, no se genera.")
        return

    #Si no existe lo generamos.
    palabras, tokens=__get_tokens(user)
    logging.info(f"{method}-START generando resumen general con {len(palabras)} palabras que son {tokens} tokens.")

    #Generamos el resumen general.
    resp = ollama_manager.call_ollama_api(model=model, system_message=system, user_message=user, temperature=temperature)

    #Convertimos salida en texto, y cambiamos el \n por \r\n.
    salida = f"{resp}"
    salida = salida.replace("\\n", "\n")

    #Genera un archivo de salida Excel con el nombre del modelo, reemplazando el : por _.
    with open(fichero_ouput_path, "w") as f:
        f.write(salida)
    logging.info(f"{method}-END fichero {fichero_ouput_path} generado.")

    return resp

In [9]:
#export 

#######
## CONSIDERANDO SOLO 1 LLAMADA por ticket
## Función Main para la generación del flujo completo. 
#
#

def main_incidencias(url, fichero, lista_campos, app="", thinking_incidencia=False, thinking_general=False, temperature=0.7, lista_modelos_incidencias=[], lista_modelos_general=[], ejecuta_todo=True, ejecuta_incidencia=True, ejecuta_propuesta=True):
    """ 
    La función procesaIncidencias realiza todo el procesamiento que se necesita:
    - Itera sobre los distintos modelos que se tienen disponibles y configurados en el ordenador. 
    - Para cada modelo realiza las siguientes operaciones
      1. Genera un resumen general de todos los tickets. Este resumen se almacena en un fichero en la url que se recibe. 
      2. Procesa uno a uno todos los tickets para generar la información. 
      3. Genera un resumen general pero a partir de la información generada, de esta forma se puede comparar con el original.

    Parametros:
    - url: URL donde se almacena el resumen general de todos los tickets.
    - fichero: Fichero Excel que contiene los tickets a procesar.   
    - lista_campos: Lista de campos que se necesitan para procesar los tickets. 
    - thinking: Booleano que indica si se debe pensar en cada paso de la función. Si es True, se imprime información detallada durante el proceso.
    
    Return:
        None
    """
    method="main_incidencias"

    logging.info(f"{method}---------------------------------------------------")
    logging.info(f"{method}-START Inicio de proceso de análisis de incidencias")
    logging.info(f"{method}- campos a procesar: {lista_campos}")
    logging.info(f"{method}- modelo análisis general: {lista_modelos_general}, razonamiento para modelos generales: {thinking_general}")
    logging.info(f"{method}- modelo análisis incidencias: {lista_modelos_incidencias}, razonamiento para modelos generales: {thinking_incidencia}")
    logging.info(f"{method}- temperatura: {temperature}, aplicación: {app}")
    logging.info(f"{method}---------------------------------------------------")

    #Leemos el fichero de propierties
    #TO-DO

    #Leemos el fichero de incidencias y generamos el listado JSON para procesar.
    listado_json, df = getIncidencias(url, fichero, lista_campos=lista_campos)

    #FASE 1 - GENERACION ANALISIS GENERAL

    #Si no se informa un listado concreto, se usan todos los disponibles.
    if lista_modelos_general == []:
        lista_modelos_general = ollama_manager.get_loaded_models()
        logging.debug(f"{method}- Tomando lista de modelos para el análisis general por defecto.")

    if ejecuta_todo:
        for model in lista_modelos_general:
            logging.info(f"{method}-Fase 1 - Inicio de proceso para el modelo {model}")
            analiza_todas_incidencias(listado_json, url, fichero, app=app, model=model, thinking=thinking_general, temperature=temperature)
    else:
        logging.info(f"{method}- ignorando la ejecución del análisis general.")

    #FASE 2 - GENERACION ANALISIS INCIDENCIAS A INCIDENCIA

    #Si no se informa un listado concreto, se usan todos los disponibles.
    if ejecuta_incidencia:
        if lista_modelos_incidencias == []:
            lista_modelos_incidencias = ollama_manager.get_loaded_models()
            logging.debug(f"{method}- Tomando todos los modelos instalados {lista_modelos_incidencias}.")
    
        for model in lista_modelos_incidencias:
            logging.info(f"{method}-Fase 2 - Inicio de proceso para el modelo {model}")
            procesa_incidencias(df, url, fichero, model, thinking=thinking_incidencia, temperature=temperature)
    else:
        logging.info(f"{method}- ignorando la ejecución incidencia a incidencia.")

    #Paso 3 - Ejecución de propuestas de mejoras y documentación para chatbot.
    if ejecuta_propuesta:
        logging.info(f"{method}- ejecutando el análisis de propuestas.")
    else:
        logging.info(f"{method}- ignorando la ejecución de propuestas.")

    logging.info(f"{method}---------------------------------------------------")
    logging.info(f"{method} END")
    logging.info(f"{method}---------------------------------------------------")


In [8]:
######################################################################
######################################################################
######################################################################

In [8]:
prompt_hranalytics=""" 
#CONTEXT.
    You are a top-tier incident analyst focused on reducing the number of tickets. 
    You are contracted to analyze the main root cause of each incident and propose a concrete fix.  
    Each incident provides three fields:
    • id: incident code.
	• desc: problem description.
	• res: resolution notes for the ticket.

#TASK.
    1. From one incident, identify the main problem(s) and propose a viable, actionable solution.
    2. Detect these groups/conditions ONLY when explicitly supported by the text:
     2.1. Problem with disk space.
     2.2. Problem accesing a la platform.
     2.3. Problem with data format between source system and the configuration of the load configurated.
     2.4. Problem with configuration or roles.
     2.5. Problem with visualization of performance.
     2.6. Problem with the daily load planned.
     2.7. Other kind of problems

#FIELD RULES (STRICT)
- Language: All JSON string values must be in English.
- Create three new fields: classification, rootcause and solution.

#CLASSIFICATION FIELD
- Create a new field called *classification* with the origen of the root cause.
- The classification field can only have the following values: Group1, Group2, Group3, Group4, Group5, Group6, Group7.
- If multiple classification could apply, choose the single most specific using the following value and order:
    -- Match the value Group1 when related with disk space. 
    -- Match the value Group2 when related with problem accesing at the platform. 
    -- Match the value Group3 when related with format problem in the interface.
    -- Match the value Group4 when related with roles or configuration. 
    -- Match the value Group5 when related with performance or visualization. 
    -- Match the value Group6 when related with planification or load processes. 
    -- Match the value Group7 when related with other problems.

#ROOTCAUSE FIELD
- Create a new field called *rootcause* with the short description of the root cause.
- The rootcause field should have a max of 10 words with a concise summary of the cause. 

#SOLUTION FIELD
- Create a new filed called *solution* with a short description of the solution for the incident. 
- The solution field should have a specific, practical action the analyst would take or request. 

#QUALITY CHECK (before finalizing)
- Ensure all new fields are written in English.
- Ensure the 3 fields are present and in the exact order.
- Ensure rootcause has 10 words or fewer.

#INPUT
The incident description is: '''{desc}'''
The resolution notes are: '''{res}'''
    
#FINAL OUTPUT FORMAT (return only this JSON; example structure shown, replace values accordingly)
{{"classification": "...","rootcause": "...","solution": "..."}}
"""


In [ ]:
prompt_webuy=""" 
#CONTEXT.
    You are a top-tier incident analyst focused on reducing the number of tickets. 
    You are contracted to analyze the main root cause of each incident and propose a concrete fix.  
    Each incident provides three fields:
    • id: incident code.
    • desc: problem description.
    • res: resolution notes for the ticket.

#TASK.
    1. From one incident, identify the main problem(s) and propose a viable, actionable solution.

    2. Based on the incident description and resolution, detect which of the following groups best matches the main root cause. 
       Only assign a group when it is explicitly supported by the text:
       2.1. Not enough incident description or incident description is not provided.
       2.2. Other incident information is not enough as tickets created without all information, wrong dates, email or cui not informed. 
       2.3. Problems accesing portal because forgetten passwords, expired passwords, or url is not able.
       2.4. Configuration errors, because incorrect roles, problem with permisos, or problem with integrated systems.
       2.5. Problem with Documentation or guide, like navigation guide, or files format.
       2.6. Onboarding state in Webuy, for example supplier remais in "Onboarding". 
       2.7. Non validated for "Compra Delegada". 
       2.8. SAP propagation issues for the supplier (e.g. supplier exists in Webuy but not in SAP, invoices cannot be validated because the supplier is not registered in SAP).
       2.9. Email or user conflicts caused by previous or archived self-registrations (e.g. email already used, need to release or delete an archived user/CUI to complete a new registration).
       2.10. Other kind of problems not related to the previous categories.

#FIELD RULES (STRICT)
- Language: All JSON string values must be in English.
- Create three new fields: classification, rootcause and solution.

#CLASSIFICATION FIELD
- Create a new field called *classification* with the origin of the root cause.
- The classification field can only have the following values: Group1, Group2, Group3, Group4, Group5, Group6, Group7, Group8, Group9, Group10.
- If multiple classification could apply, choose the single most specific using the following value and order:
    -- Match the value Group1 when the issue is related to not enough incident description or incident description is not provided.
    -- Match the value Group2 when the issue is related to other incident information is not enough as tickets created without all information, wrong dates, email or cui not informed. 
    -- Match the value Group3 when the issue is related to problems accesing portal because forgetten passwords, expired passwords, or url is not able.
    -- Match the value Group4 when the issue is related to configuration errors, because incorrect roles, problem with permisos, or problem with integrated systems.
    -- Match the value Group5 when the issue is related to problem with Documentation or guide, like navigation guide, or files format.
    -- Match the value Group6 when the issue is related to onboarding state in Webuy, for example supplier remais in "Onboarding"
    -- Match the value Group7 when the issue is related to non validated for "Compra Delegada". 
    -- Match the value Group8 when the issue is related to SAP propagation issues for the supplier (e.g. supplier exists in Webuy but not in SAP, invoices cannot be validated because the supplier is not registered in SAP).
    -- Match the value Group9 when the issue is related to email or user conflicts caused by previous or archived self-registrations (e.g. email already used, need to release or delete an archived user/CUI to complete a new registration).
    -- Match the value Group10 when the issue is related to other kind of problems not related to the previous categories.

#ROOTCAUSE FIELD
- Create a new field called *rootcause* with the short description of the root cause.
- The rootcause field should have a max of 10 words with a concise summary of the cause. 

#SOLUTION FIELD
- Create a new filed called *solution* with a short description of the solution for the incident. 
- The solution field should have a specific, practical action the analyst would take or request. 

#QUALITY CHECK (before finalizing)
- Ensure all new fields are written in English.
- Ensure the 3 fields are present and in the exact order.
- Ensure rootcause has 10 words or fewer.

#INPUT
The incident description is: '''{desc}'''
The resolution notes are: '''{res}'''
    
#FINAL OUTPUT FORMAT (return only this JSON; example structure shown, replace values accordingly)
{{"classification": "...","rootcause": "...","solution": "..."}}
"""

In [13]:

url_windows="C:/Users/Es44031665m/Downloads"

usuario = 'Users/zzddfge'
carpeta = 'trabajo/incidencias'
disco_duro = '/Volumes/Macintosh HD'

url_mac=os.path.join(disco_duro, usuario, carpeta)


fichero_onboarding="incidencias_onboarding.xlsx"
fichero_analytics="incidencias_hranalytics.xlsx"
fichero_shemetrics="incidencias_shemetrics.xlsx"
fichero_webuy="incidencias_webuy.xlsx"

#lista_campos_onboarding=["Número", "Descripción corta", "Notas de resolución"]
lista_campos_espanol=["Número", "Descripción", "Notas de resolución"]
lista_campos_italiano=["Numero", "Descrizione", "Note sulla risoluzione"]

lista_modelos_general=["qwen3:30b-a3b-thinking-2507-q4_K_M"]
lista_modelos_incidencias=["qwen3:30b-a3b-thinking-2507-q4_K_M"]

app="WeBuy"

logging.root.handlers.clear()
logging.basicConfig(format='%(asctime)s-%(levelname)s-  %(message)s', level=logging.INFO)

class HttpRequestFilter(logging.Filter):
    def filter(self, record):
        # Comprueba si el mensaje contiene la cadena específica del HTTP request
        return "HTTP Request" not in record.getMessage()

for handler in logging.getLogger().handlers:
    handler.addFilter(HttpRequestFilter())

__set_prompt(PROMPT_INCIDENCIAS, prompt_webuy)

fichero = "incidencias_webuy.xlsx"
lista_campos = lista_campos_espanol
#"qwen3:4b-thinking-2507-q4_K_M", 

lista_modelos_incidencias=["mistral-small3.2:latest", "qwen3:30b-a3b-thinking-2507-q4_K_M"]
lista_modelos_incidencias=["ministral-3:14b", "gemma3:27b"]
lista_modelos_incidencias=["nemotron-3-nano:latest"]


main_incidencias(url_mac, fichero, lista_campos, app=app,
                 thinking_general=True, thinking_incidencia=True, temperature=0.6,
                 lista_modelos_incidencias=lista_modelos_incidencias, lista_modelos_general=lista_modelos_general,
                 ejecuta_todo=False, ejecuta_incidencia=True, ejecuta_propuesta=False
                )


2025-12-17 23:38:39,904-INFO-  main_incidencias---------------------------------------------------
2025-12-17 23:38:39,905-INFO-  main_incidencias-START Inicio de proceso de análisis de incidencias
2025-12-17 23:38:39,905-INFO-  main_incidencias- campos a procesar: ['Número', 'Descripción', 'Notas de resolución']
2025-12-17 23:38:39,906-INFO-  main_incidencias- modelo análisis general: ['qwen3:30b-a3b-thinking-2507-q4_K_M'], razonamiento para modelos generales: True
2025-12-17 23:38:39,906-INFO-  main_incidencias- modelo análisis incidencias: ['nemotron-3-nano:latest'], razonamiento para modelos generales: True
2025-12-17 23:38:39,906-INFO-  main_incidencias- temperatura: 0.6, aplicación: WeBuy
2025-12-17 23:38:39,906-INFO-  main_incidencias---------------------------------------------------
2025-12-17 23:38:39,907-INFO-  getIncidencias-START - Leyendo el archivo Excel: /Volumes/Macintosh HD/Users/zzddfge/trabajo/incidencias/incidencias_webuy.xlsx
2025-12-17 23:38:40,028-INFO-  getInci

KeyboardInterrupt: 

In [42]:
listado_json, df = getIncidencias(url_mac, fichero, lista_campos=lista_campos)
print(listado_json)

2025-12-06 00:33:12,443-INFO-  getIncidencias-START - Leyendo el archivo Excel: /Volumes/Macintosh HD/Users/zzddfge/trabajo/incidencias/incidencias_webuy.xlsx
2025-12-06 00:33:12,503-INFO-  getIncidencias- 575 registros leídos.
2025-12-06 00:33:12,505-DEBUG-  getIncidencias-END - Se ha filtrado el DataFrame para obtener los campos especificados: Index(['id', 'desc', 'res', 'status', 'cot'], dtype='object')


[{"id":"INC000171998199","desc":"la empresa con CIF B09875139, se ha dado de alta en la plataforma Webuy, parece todo correcto, pero no termina de migrar en SAP.\n\n Contact People:ES80101743B\n Name:JORGE GARCIA MONTE\n Email:JORGE.GARCIAMONTE@ENEL.COM\n Location:Spain-SAN ROQUE-PG GUADARRANQUE 6_7\n Business phone:+34 653 65 55 81\n Alternative Telephone: +34 653 65 55 81\n Time Slot: Anytime\n Alternative Email: JORGE.GARCIAMONTE@ENEL.COM","res":"RECAP : The user requests that the Self-registered supplier be archived in WeBuy to perform the registration as Delegated Purchasing\nANALYSIS : The Supplier's status is Complete.\nRESOLUTION : As requested, the CUI has been Archived"},{"id":"INC000172840326","desc":"NAME: Fernando - COMPANY: 14382585000166 - PHONE: +5522997468071 - DESCRIPTION: Prezado(a), registrei no site, porem n\u00e3o sei como devo prosseguir para poder obter sucesso em oferecer meus servi\u00e7os a Enel atrav\u00e9s do site. N\u00e3o conhe\u00e7o ningu\u00e9m dentro 

In [12]:
!pip install httpx


[notice] A new release of pip is available: 25.0.1 -> 25.2
[notice] To update, run: pip install --upgrade pip


In [ ]:
###############################################################################
###############################################################################
###############################################################################

In [3]:
#revisión del paso 2 y 3
# Paso 2 --> a partir de las incidencias proponer mejoras de la aplicación
# Paso 3 --> a partir de las incidencias generar documentación que sirva para reducir los tickets en un sistema de RAG.


In [10]:
#Paso 3 --> A partir de als incidencias generar documentación

url_windows="C:/Users/Es44031665m/Downloads"

usuario = 'Users/zzddfge'
carpeta = 'trabajo/incidencias'
disco_duro = '/Volumes/Macintosh HD'

url_mac=os.path.join(disco_duro, usuario, carpeta)


fichero_onboarding="incidencias_onboarding.xlsx_qwen3_30b-a3b-thinking-2507-q4_K_M.xlsx"
fichero_analytics="incidencias_hranalytics.xlsx_qwen3_30b-a3b-thinking-2507-q4_K_M.xlsx"
fichero_webuy = "incidencias_webuy.xlsx_qwen3_30b-a3b-thinking-2507-q4_K_M.xlsx"

#lista_campos_onboarding=["Número", "Descripción corta", "Notas de resolución"]
lista_campos_onboarding=["root_cause", "solution"]
lista_campos_webuy=["Número", "Descripción", "Notas de resolución"]

lista_modelos_general=["qwen3:30b-a3b-thinking-2507-q4_K_M"]
lista_modelos_incidencias=["qwen3:30b-a3b-thinking-2507-q4_K_M"]

lista_modelos_general=["nemotron-3-nano:latest"]
lista_modelos_incidencias=["nemotron-3-nano:latest"]

app="ANALYTICS"

logging.root.handlers.clear()
logging.basicConfig(format='%(asctime)s-%(levelname)s-  %(message)s', level=logging.INFO)

class HttpRequestFilter(logging.Filter):
    def filter(self, record):
        # Comprueba si el mensaje contiene la cadena específica del HTTP request
        return "HTTP Request" not in record.getMessage()

for handler in logging.getLogger().handlers:
    handler.addFilter(HttpRequestFilter())

method="TODO"


fichero = fichero_webuy
file_name_path=os.path.join(disco_duro, usuario, carpeta, fichero)
logging.info(f"{method}- START con {file_name_path}")

df = pd.read_excel(file_name_path, sheet_name=0)
logging.info(f"{method}- {df.shape[0]} registros leídos.")

#Copia y selección de los tres campos que necesitamos.
df_filtrado = df.copy()
df_filtrado = df_filtrado[lista_campos_onboarding]
df_filtrado.columns = df_filtrado.columns.str.lower()

#Creación del listado json antes de añadirle el campo status.
listado_json = df_filtrado.to_json(orient='records')



2025-12-17 23:36:55,750-INFO-  TODO- START con /Volumes/Macintosh HD/Users/zzddfge/trabajo/incidencias/incidencias_webuy.xlsx_qwen3_30b-a3b-thinking-2507-q4_K_M.xlsx


FileNotFoundError: [Errno 2] No such file or directory: '/Volumes/Macintosh HD/Users/zzddfge/trabajo/incidencias/incidencias_webuy.xlsx_qwen3_30b-a3b-thinking-2507-q4_K_M.xlsx'

In [58]:
# Convertir el JSON de cadena a una lista de diccionarios
import json
data_list = json.loads(listado_json)
len_elementos=len(data_list)

In [59]:
APP="OnBoarding"
APP="Analytics"
APP="WeBuy"

system_prompt=""" 
        Eres un analista de proceso encargado de analizar listas de incidencias, agrupar aquellas que son comunes 
        en análisis semántico y analizar los distintos tipos de soluciones para identificar la información necesaria 
        que debe guiar a un sistema automático para preguntar a un usuario y evitar que se genere un ticket. 
        
        Deberás considerar los mismos conceptos bajo la misma causa raíz, por ejemplo:
        - mail inexistente, correo electrónico no introducido, son la misma causa raíz. 

        Eres preciso con la información que se tiene que generar. 
"""
user_prompt=f""" 
#CONTEXTO.
    Hay un sistema, llamado Tell Me, que es una arquitectura de RAG, integrado en el proceso de apertura de 
    tickets. En Tell Me se ingesta documentación para guiar al usuario en el proceso de apertura, de forma que se pueda reducir el número de tickets. En la actualidad tenemos un 30% de reducción
    de apertura de tickets.

    Tenemos un listado de incidencias de la aplicación {APP} que han sido analizados, generándose los campos:
    * root_cause: Es la causa raíz de la incidencia. 
    * solution: Es el resumen de la solución que se ha dado al usuario. 

    Te han contratado con el rol es de analista de procesos, tienes en el contrato una prima de 2000$ si logras 
    aumentar la reducción desde el 30% al 60%. Si no haces un buen trabajo, a mi me despedirán y tú no tendrás ningún 
    pago. 

    #FORMATO DE ENTRADA (JSON)
    Lista de objetos con 
	• root_cause: causa raíz de la incidencia.
	• solution: resumen de la solución del ticket.
	
    Example input (referred later as incident_list_json:
    [{{"root_cause":"...","solution":"..."}},{{"root_cause":"...","solution":"..."}}]
    
    #TAREA
	* Agrupa cada causa raíz por simulitud semántico.
	* Extrae de las solution la información exacta y necesaria que un chatbot debe entregar para que el usuario resuelva cada tipo de incidencia. 
    * Genera la documentación para el sistema RAG, organizada por los grupos de causas raíz.
    * Propón mejoras en los sistemas que reduzcan la recurrencia de estos problemas.

    #INSTRUCCIONES
    * Usa exclusivamente la información del input. No inventes datos. 
    * Elimina duplicidades y evita contradicciones.
    * La agrupación debe hacerse solo por significado operativo, no por similitud superficial de palabras.
    * Asigna a cada grupo un nombre corto y consistente, y úsalo en toda la salida.
    * Las propuestas de mejora deben estar claramente vinculadas a los grupos de causas raíz afectados.
    * Responde siempre en español con redacción clara y útil para su incorporación a un RAG.
    

    #FORMATO DE SALIDA REQUERIDO
    1. **NÚMERO DE INCIDENCIAS**
	* Indica cuántos elementos contiene la lista y explica brevemente el método de conteo.

	2.	**DOCUMENTACIÓN PARA RAG**
	* Presenta la información consolidada y sin duplicados, organizada por cada grupo de causas raíz (usar el nombre del grupo como encabezado).
	* Para cada grupo:
	    - Resumen de la causa raíz.
	    - Información que el chatbot debe comunicar al usuario para resolverla, basada en las solution originales.
    * Usa este formato para la documentación:
        - Usa un tono formal, amigable, y directo. 
        - Desarrolla el texto manteniendo una introducción seguida de una serie de acciones a realizar. 
        - Tanto la parte de introducción como cada una de las reglas debe tener una longitud de al menos 3 líneas. 

	3. **PROPUESTAS DE MODIFICACIÓN EN SISTEMAS**
	* Cambios necesarios en los sistemas origen.
	* Cada cambio debe indicar explícitamente a qué grupo de causa raíz beneficia.
    
	4. **CONCLUSIONES**
	* Hallazgos clave y recomendaciones para la reducción de incidencias.
    * Deben ser conclusiones útiles para tu jefe y para el usuario.
    
    #VERIFICACIÓN
    * Validar que toda la salida es veraz respecto al input.
    * Verificar ausencia de incoherencias o información contradictoria.
    * Confirmar que la salida está en español.
    * Asegurar que las conclusiones explican claramente qué está causando el volumen de incidencias.
    * En concreto el listado tiene {len_elementos} elementos. Verifica que cuadra con la información que has incluido. 
    
    #INPUT
    {listado_json}
"""

In [60]:
user_prompt

' \n#CONTEXTO.\n    Hay un sistema, llamado Tell Me, que es una arquitectura de RAG, integrado en el proceso de apertura de \n    tickets. En Tell Me se ingesta documentación para guiar al usuario en el proceso de apertura, de forma que se pueda reducir el número de tickets. En la actualidad tenemos un 30% de reducción\n    de apertura de tickets.\n\n    Tenemos un listado de incidencias de la aplicación WeBuy que han sido analizados, generándose los campos:\n    * root_cause: Es la causa raíz de la incidencia. \n    * solution: Es el resumen de la solución que se ha dado al usuario. \n\n    Te han contratado con el rol es de analista de procesos, tienes en el contrato una prima de 2000$ si logras \n    aumentar la reducción desde el 30% al 60%. Si no haces un buen trabajo, a mi me despedirán y tú no tendrás ningún \n    pago. \n\n    #FORMATO DE ENTRADA (JSON)\n    Lista de objetos con \n\t• root_cause: causa raíz de la incidencia.\n\t• solution: resumen de la solución del ticket.\n\n

In [61]:
palabras, tokens = __get_tokens(user_prompt)
logging.info(f"{method}- Prompt con {len(palabras)} palabras. Tokens estimados son: {tokens}")

2025-12-06 08:33:39,124-INFO-  TODO- Prompt con 6384 palabras. Tokens estimados son: 8619


In [ ]:

model = lista_modelos_general[0]
#model="gpt-oss:latest"
logging.info(f"{method}- model={model}")

#Generamos el resumen general.
resp = ollama_manager.call_ollama_api(model=model, system_message=system_prompt, user_message=user_prompt, temperature=0.6, max_token=1000000, think=True)

#Convertimos salida en texto, y cambiamos el \n por \r\n.
salida = f"{resp}"
salida = salida.replace("\\n", "\n")
logging.info(salida)
  

In [10]:
resp

'1. **NÚMERO DE INCIDENCIAS**  \nSe analizaron 392 incidencias, correspondientes al listado proporcionado. El conteo se validó contando los elementos en el JSON de entrada, excluyendo duplicados semánticos identificados durante el agrupamiento (ej.: múltiples referencias a "correo electrónico incorrecto" se consolidaron en un solo grupo). La verificación confirmó que todos los elementos del input fueron procesados, incluyendo el último registro nulo, que fue descartado como dato inválido.\n\n---\n\n2. **DOCUMENTACIÓN PARA RAG**  \n\n**Correo electrónico incorrecto o faltante en sistema**  \n*Resumen:* El correo corporativo no coincide con el registrado, está incompleto o no se actualizó, impidiendo acceso al portal, notificaciones y procesos de onboarding.  \n*Chatbot:*  \nVerifica que tu correo electrónico corporativo esté registrado en el sistema como @enel.com. Si no es correcto, actualízalo en el portal de onboarding o solicita a RRHH que lo corrija. Asegúrate de no usar correos pe

In [22]:
model="gpt-oss:latest"
#model="qwen3:30b-a3b-thinking-2507-q4_K_M"

import httpx
timeout = httpx.Timeout(
            connect=60.0,      # conexión: 1 minuto
            read=3600.0,       # lectura de respuesta: 60 minutos
            write=60.0,        # escritura de payload: 10 minuto
            pool=None          # sin timeout en la conexión del pool
        )

api_base="http://localhost:11434/v1"
api_key="ollama"

client = OpenAI(
            base_url = api_base,
            api_key = api_key,
            http_client=httpx.Client(timeout=timeout)
        )

messages=[
            {"role":"system", "content": system_prompt},
            {"role": "user", "content": user_prompt}
        ]
                                 
response=client.chat.completions.create(
            model=model, 
            messages=messages,
            temperature=0.6,
            extra_body={"think": False},
            max_tokens=1000000)

response.choices[0]

Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content='**1. NÚMERO DE INCIDENCIAS**  \n- La lista contiene **392** objetos.  \n- Se obtuvo contando el número de elementos del array JSON proporcionado (`len(incident_list_json)`), que incluye todas las incidencias registradas, incluso la última entrada con valores nulos que marca el final de la lista.  \n\n---\n\n## 2. DOCUMENTACIÓN PARA RAG  \n\nA continuación se presentan los grupos de causas raíz, cada uno con una introducción formal y amigable, seguida de las acciones concretas que el chatbot debe comunicar al usuario. Los nombres de los grupos son cortos y consistentes, y se utilizan en todas las secciones.  \n\n### **AccesoPortal**  \n**Resumen de la causa raíz**  \nMuchas incidencias indican que el usuario no puede acceder al portal de onboarding debido a permisos insuficientes, cuentas desactivadas o configuraciones incorrectas.  \n\n**Información que el chatbot debe comunicar**  \n1. *Introdu

In [24]:
try:
    reasoning = response.choices[0].message.reasoning
except Exception as e:
    reasoning = None 
content = response.choices[0].message.content 

salida = f"<think>{reasoning}</think> {content}" if reasoning is not None  else content
salida

'<think>We need to produce output. There\'s 392 elements. Must group by root_cause semantics. But we have huge list. We cannot manually group all. But maybe we can summarize? The instruction: "Agrupa cada causa raíz por similitud semántico." We need to produce documentation for RAG organized by groups. This is huge. But maybe we can produce a concise set of groups covering main categories: Access issues, Email issues, Document upload issues, Task workflow issues, etc. We need to extract from solutions the exact info for chatbot. Provide groups names short and consistent. Also proposals of modification.\n\nWe must ensure output matches 392 count. We need to count elements. The input ends with {"root_cause":null,"solution":null}. That indicates end? The list has 392 elements. We must mention that. We must explain method of counting: length of array.\n\nWe need to produce documentation with group headings and info. We need to avoid contradictions. Provide a concise but thorough.\n\nWe can